<a href="https://colab.research.google.com/github/Nikhil4002-50-82/Automatic-Pedicle-Screw-Planning/blob/main/SpineSegementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***For Single Data Segementation***

In [1]:
!pip -q install totalsegmentator nibabel SimpleITK

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.9/212.9 kB 9.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import files
uploaded = files.upload()

Saving case_0000.nii to case_0000.nii


In [11]:
import glob

ct_files = sorted(glob.glob("/content/*.nii*"))
print("Found:", ct_files)

if len(ct_files) == 0:
    raise RuntimeError("No CT .nii/.nii.gz uploaded!")

CT_PATH = ct_files[0]
print("Using CT:", CT_PATH)

Found: ['/content/case_0000.nii']
Using CT: /content/case_0000.nii


In [12]:
LICENSE_ID = "aca_V4E9OLA5VBGIQO"
!totalseg_set_license -l "$LICENSE_ID"

License has been successfully saved.


In [13]:
OUT_DIR = "/content/totalseg_out"
!rm -rf "$OUT_DIR"
!mkdir -p "$OUT_DIR"

!TotalSegmentator -i "$CT_PATH" -o "$OUT_DIR" -ta total -ml -rs vertebrae_L1 vertebrae_L2 vertebrae_L3 vertebrae_L4 vertebrae_L5


If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

TotalSegmentator sends anonymous usage statistics. If you want to disable it check the documentation.
Downloading: 100% 234M/234M [00:11<00:00, 20.3MB/s]
Download finished. Extracting...
Downloading: 100% 234M/234M [00:08<00:00, 26.4MB/s]
Download finished. Extracting...
Downloading: 100% 234M/234M [00:53<00:00, 4.38MB/s]
Download finished. Extracting...
Downloading: 100% 234M/234M [00:08<00:00, 26.8MB/s]
Download finished. Extracting...
Downloading: 100% 234M/234M [00:31<00:00, 7.42MB/s]
Download finished. Extracting...
Generating rough segmentation for cropping...
Downloading: 100% 135M/135M [00:18<00:00, 7.44MB/s]
Download finished. Extracting...
Resampling...
  Resampled in 2.21s
Predicting...
100% 12/12 [00:01<00:00,  8.93it/s]
  Predicted in 15.39s
Resampling...
  cropping from (279, 279, 490) to (95, 99, 147)
Resampling...
  Resampled in 0.00s
Predicting part 1 of 1 ...
100% 2/2 [00:01<00:00,  1.16

In [17]:
import glob, os

out_files = sorted(glob.glob("/content/*"))
print("Output files:")
for f in out_files:
    print(os.path.basename(f))

Output files:
case_0000.nii
sample_data
totalseg_out
totalseg_out.nii


In [19]:
import nibabel as nib
import numpy as np

SEG_PATH = "/content/totalseg_out.nii"

seg_nii = nib.load(SEG_PATH)
seg = seg_nii.get_fdata().astype(np.int16)

print("Seg shape:", seg.shape)
print("Unique labels:", np.unique(seg))

Seg shape: (279, 279, 490)
Unique labels: [ 0 27 28 29 30 31]


In [20]:
labels = sorted([x for x in np.unique(seg) if x != 0])
print("Non-zero labels found:", labels)

if len(labels) != 5:
    print("WARNING: Expected 5 labels, but found", len(labels))

new_seg = np.zeros_like(seg, dtype=np.uint8)

for new_lab, old_lab in enumerate(labels, start=1):
    new_seg[seg == old_lab] = new_lab

print("Unique labels after remap:", np.unique(new_seg))

Non-zero labels found: [np.int16(27), np.int16(28), np.int16(29), np.int16(30), np.int16(31)]
Unique labels after remap: [0 1 2 3 4 5]


In [21]:
OUT_FIXED = "/content/spine_L1_L5_seg_fixed.nii.gz"

nib.save(nib.Nifti1Image(new_seg, seg_nii.affine, seg_nii.header), OUT_FIXED)
print("Saved:", OUT_FIXED)

Saved: /content/spine_L1_L5_seg_fixed.nii.gz


# ***For Batch Data Segmentation***

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil
from google.colab import files
import nibabel as nib
import numpy as np

print("Upload all your CT .nii/.nii.gz files:")
uploaded = files.upload()

uploaded_files = list(uploaded.keys())
print(f"Uploaded {len(uploaded_files)} files: {uploaded_files}")

BASE_DIR = "/content/drive/MyDrive/SpineData"
os.makedirs(BASE_DIR, exist_ok=True)

LICENSE_ID = "aca_V4E9OLA5VBGIQO"
!totalseg_set_license -l "$LICENSE_ID"

for idx, fname in enumerate(uploaded_files, start=1):
    case_id = f"case_{idx:03d}"
    case_dir = os.path.join(BASE_DIR, case_id)
    os.makedirs(case_dir, exist_ok=True)

    ct_target_path = os.path.join(case_dir, "case_0000.nii")
    shutil.copy(fname, ct_target_path)
    print(f"[{case_id}] Saved original CT as {ct_target_path}")

    seg_out_dir = os.path.join(case_dir, "totalseg_out")
    os.makedirs(seg_out_dir, exist_ok=True)

    print(f"[{case_id}] Running TotalSegmentator...")
    !TotalSegmentator -i "{ct_target_path}" -o "{seg_out_dir}" -ta total -ml -rs vertebrae_L1 vertebrae_L2 vertebrae_L3 vertebrae_L4 vertebrae_L5

    seg_files = glob.glob(os.path.join(seg_out_dir, "*.nii*"))
    if len(seg_files) == 0:
        print(f"[{case_id}] WARNING: No segmentation file found!")
        continue
    seg_path = seg_files[0]

    seg_nii = nib.load(seg_path)
    seg = seg_nii.get_fdata().astype(np.int16)

    labels = sorted([x for x in np.unique(seg) if x != 0])
    new_seg = np.zeros_like(seg, dtype=np.uint8)
    for new_lab, old_lab in enumerate(labels, start=1):
        new_seg[seg == old_lab] = new_lab

    out_fixed = os.path.join(case_dir, "spine_2-label.nii")
    nib.save(nib.Nifti1Image(new_seg, seg_nii.affine, seg_nii.header), out_fixed)
    print(f"[{case_id}] Saved fixed segmentation as {out_fixed}")

    shutil.rmtree(seg_out_dir)

print("\nAll files processed and stored in SpineData/")

Mounted at /content/drive
Upload all your CT .nii/.nii.gz files:


Saving case_0000.nii to case_0000.nii
Saving case_0007.nii to case_0007.nii
Saving case_0014.nii to case_0014.nii
Saving case_0020.nii to case_0020.nii
Saving case_0025.nii to case_0025.nii
Saving case_0037.nii to case_0037.nii
Saving case_0067.nii to case_0067.nii
Saving case_0078.nii to case_0078.nii
Saving case_0079.nii to case_0079.nii
Saving case_0468.nii to case_0468.nii
Saving case_0511.nii to case_0511.nii
Uploaded 11 files: ['case_0000.nii', 'case_0007.nii', 'case_0014.nii', 'case_0020.nii', 'case_0025.nii', 'case_0037.nii', 'case_0067.nii', 'case_0078.nii', 'case_0079.nii', 'case_0468.nii', 'case_0511.nii']
License has been successfully saved.
[case_001] Saved original CT as /content/drive/MyDrive/SpineData/case_001/case_0000.nii
[case_001] Running TotalSegmentator...

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

TotalSegmentator sends anonymous usage statistics. If you want to disable it check the documentation.
Downloading: 100% 234M/234M

In [3]:
import glob
glob.glob("/content/drive/MyDrive/SpineData/case_001/*.nii*")

['/content/drive/MyDrive/SpineData/case_001/case_0000.nii',
 '/content/drive/MyDrive/SpineData/case_001/totalseg_out.nii']

In [4]:
import os
import glob
import nibabel as nib
import numpy as np

BASE_DIR = "/content/drive/MyDrive/SpineData"

case_folders = sorted(glob.glob(os.path.join(BASE_DIR, "case_*")))

for case_dir in case_folders:
    totalseg_file = os.path.join(case_dir, "totalseg_out.nii")
    if not os.path.exists(totalseg_file):
        print(f"[{case_dir}] WARNING: totalseg_out.nii not found!")
        continue

    seg_nii = nib.load(totalseg_file)
    seg = seg_nii.get_fdata().astype(np.int16)

    # Get unique non-zero labels (L1-L5)
    labels = sorted([x for x in np.unique(seg) if x != 0])
    new_seg = np.zeros_like(seg, dtype=np.uint8)

    for new_lab, old_lab in enumerate(labels, start=1):
        new_seg[seg == old_lab] = new_lab

    out_file = os.path.join(case_dir, "spine_2-label.nii")
    nib.save(nib.Nifti1Image(new_seg, seg_nii.affine, seg_nii.header), out_file)
    print(f"[{case_dir}] Saved spine_2-label.nii")

[/content/drive/MyDrive/SpineData/case_001] Saved spine_2-label.nii
[/content/drive/MyDrive/SpineData/case_002] Saved spine_2-label.nii
[/content/drive/MyDrive/SpineData/case_003] Saved spine_2-label.nii
[/content/drive/MyDrive/SpineData/case_004] Saved spine_2-label.nii
[/content/drive/MyDrive/SpineData/case_005] Saved spine_2-label.nii
[/content/drive/MyDrive/SpineData/case_006] Saved spine_2-label.nii
[/content/drive/MyDrive/SpineData/case_007] Saved spine_2-label.nii
[/content/drive/MyDrive/SpineData/case_008] Saved spine_2-label.nii
[/content/drive/MyDrive/SpineData/case_009] Saved spine_2-label.nii
[/content/drive/MyDrive/SpineData/case_010] Saved spine_2-label.nii
[/content/drive/MyDrive/SpineData/case_011] Saved spine_2-label.nii
